In [1]:
import pandas as pd
from chembl_webresource_client.new_client import new_client
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from tqdm import tqdm

print('Imports OK')

Imports OK


In [3]:
molecule = new_client.molecule

print('Querying ChEMBL, please wait...')

results = molecule.filter(
    molecule_type='Small molecule',
    structure_type='MOL',
    molecule_properties__mw_freebase__gte=100,
    molecule_properties__mw_freebase__lte=250,
    molecule_properties__rtb__lte=5,
    molecule_properties__alogp__lte=5,
    molecule_properties__hbd__lte=5,
    molecule_properties__hba__lte=10,
).only([
    'molecule_chembl_id',
    'molecule_structures',
    'molecule_properties'
])[:10000]

results_list = list(results)
print(f'Downloaded {len(results_list)} compounds.')

Querying ChEMBL, please wait...
Downloaded 10000 compounds.


In [4]:
records = []

for mol in results_list:
    chembl_id = mol.get('molecule_chembl_id', None)
    structures = mol.get('molecule_structures', None)
    if structures is None:
        continue
    smiles = structures.get('canonical_smiles', None)
    if smiles is None:
        continue
    props = mol.get('molecule_properties', None)
    if props is None:
        continue
    records.append({
        'chembl_id': chembl_id,
        'smiles': smiles,
        'mw': props.get('mw_freebase'),
        'alogp': props.get('alogp'),
        'hbd': props.get('hbd'),
        'hba': props.get('hba'),
        'rtb': props.get('rtb'),
    })

df = pd.DataFrame(records)
print(f'Parsed {len(df)} compounds.')
df.head()

Parsed 10000 compounds.


,chembl_id,smiles,mw,alogp,hbd,hba,rtb
0,CHEMBL268150,CC(=O)c1ccc(-n2ncc(=O)[nH]c2=O)cc1,231.21,0.12,1,5,2
1,CHEMBL204146,O=c1oc2c(O)c(O)ccc2c2ccc(O)cc12,244.20,2.06,3,5,0
2,CHEMBL6355,O=c1oc2ccccc2c2ccccc12,196.21,2.95,0,2,0
3,CHEMBL6364,O=c1oc2c(O)c(O)ccc2c2ccccc12,228.20,2.36,2,4,0
4,CHEMBL6365,O=c1oc2ccccc2c2cc(O)c(O)cc12,228.20,2.36,2,4,0


rdkit filtering


In [5]:
ALLOWED_ATOMS = {6, 1, 7, 8, 16, 15, 9, 17, 35, 53}
# C, H, N, O, S, P, F, Cl, Br, I

def passes_rdkit_filters(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False, None
    # Must contain carbon
    atom_nums = {atom.GetAtomicNum() for atom in mol.GetAtoms()}
    if 6 not in atom_nums:
        return False, None
    # Only allowed elements
    if not atom_nums.issubset(ALLOWED_ATOMS):
        return False, None
    # No isotopes
    if any(atom.GetIsotope() != 0 for atom in mol.GetAtoms()):
        return False, None
    return True, mol

passed_smiles = []
passed_chembl = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Filtering'):
    ok, mol = passes_rdkit_filters(row['smiles'])
    if ok:
        passed_smiles.append(row['smiles'])
        passed_chembl.append(row['chembl_id'])

df_filtered = pd.DataFrame({
    'chembl_id': passed_chembl,
    'smiles': passed_smiles
})

print(f'Before RDKit filtering: {len(df)} compounds')
print(f'After RDKit filtering:  {len(df_filtered)} compounds')

Filtering: 100%|██████████| 10000/10000 [00:05<00:00, 1680.49it/s]

Before RDKit filtering: 10000 compounds
After RDKit filtering:  9945 compounds


In [6]:
output_path = 'chembl_filtered_for_standardization.smi'

with open(output_path, 'w') as f:
    for _, row in df_filtered.iterrows():
        f.write(f"{row['smiles']} {row['chembl_id']}\n")

print(f'Exported {len(df_filtered)} SMILES to {output_path}')
print(f'File saved to: {output_path}')

Exported 9945 SMILES to chembl_filtered_for_standardization.smi
File saved to: chembl_filtered_for_standardization.smi
